# 第五讲 动手：动作模式与每步上限

贯通实验第四步：同一个目标，两种动作模式。先依次运行下面七格「准备」（导入库，以及上一讲写好的正逆运动学），再从第一段往下，每格运行一次。每一步的输出和仿真给的真值放在一起对答案；实验记录存在 `out/05-modes.json`。

In [ ]:
# 准备（一）：导入库
import json                                                        # 存实验结果
import xml.etree.ElementTree as ET                                 # 读 URDF（XML 文件）
from pathlib import Path                                           # 路径

import gymnasium as gym                                            # 仿真环境的标准接口
import matplotlib.pyplot as plt                                    # 画图
import numpy as np                                                 # 数组计算
import so101_sim                                                   # 导入时注册星禾仿真的三个场景
from so101_sim.robots.so101_kit_slow import REAL_P95_DEG_PER_S, SIM_FPS, SPEED_SCALE, per_step_limits  # 真机实测速度、控制频率、放慢系数

In [ ]:
# 准备（二）：画图字体，场景、结果目录、关节名与机器人模型文件的路径
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "DejaVu Sans"]  # 图里的中文用思源黑体
plt.rcParams["axes.unicode_minus"] = False                         # 负号照常显示
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())  # 动手目录（有 pyproject.toml 的那一层）
OUT = HERE / "out"                                                 # 实验结果存在 out/ 里
ENV = "SO101PickPlaceCube40-v1"                                    # 星禾仿真的场景：把 4 厘米方块放进料盒
JOINTS = list(REAL_P95_DEG_PER_S)                                  # 六个关节的名字：五个臂关节加夹爪
URDF = Path(so101_sim.__file__).parent / "robots/kit_assets/kit_v1_so101.urdf"  # 机器人模型文件

In [ ]:
# 准备（三）：读机器人模型文件，得到运动链与关节限位
def floats(text):                                                  # "0 0 1" → 数组
    return np.array([float(v) for v in text.split()])  # 按空格拆开，转成浮点数


chain = {}                                                         # 子连杆 → 把它接到父连杆上的那个关节
for j in ET.parse(URDF).getroot().findall("joint"):                # 逐个关节
    origin, axis, limit = j.find("origin"), j.find("axis"), j.find("limit")  # 固定变换、转轴、限位
    chain[j.find("child").get("link")] = dict(  # 记下它的父连杆、关节名、类型、固定变换、转轴与限位
        parent=j.find("parent").get("link"), name=j.get("name"), kind=j.get("type"),  # 接在哪个连杆上
        xyz=floats(origin.get("xyz", "0 0 0")), rpy=floats(origin.get("rpy", "0 0 0")),  # 固定的位置与朝向
        axis=None if axis is None else floats(axis.get("xyz")),    # 转轴
        lo=None if limit is None else float(limit.get("lower")),   # 关节下限（弧度）
        hi=None if limit is None else float(limit.get("upper")))   # 关节上限（弧度）
by_name = {j["name"]: j for j in chain.values()}                   # 同一批关节，按关节名查
LIMITS = np.array([[by_name[n]["lo"], by_name[n]["hi"]] for n in JOINTS])  # 六个关节的限位（弧度）

In [ ]:
# 准备（四）：两种旋转矩阵：固定朝向 rpy，绕转轴转一个角度
def rot_rpy(roll, pitch, yaw):                                     # 固定朝向 rpy → 旋转矩阵（先绕 x，再绕 y，最后绕 z）
    cr, sr, cp, sp, cy, sy = np.cos(roll), np.sin(roll), np.cos(pitch), np.sin(pitch), np.cos(yaw), np.sin(yaw)  # 三个角的余弦与正弦
    rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])          # 绕 x 转
    ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])          # 绕 y 转
    rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])          # 绕 z 转
    return rz @ ry @ rx                                            # 连乘

def rot_axis(axis, angle):                                         # 绕转轴转 angle 的旋转矩阵（罗德里格斯公式）
    k = axis / np.linalg.norm(axis)                                # 单位转轴
    kx = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])  # 叉乘矩阵
    return np.eye(3) + np.sin(angle) * kx + (1 - np.cos(angle)) * kx @ kx  # 单位阵加上 sinθ 项和 (1−cosθ) 项

In [ ]:
# 准备（五）：正运动学（一）：沿运动链连乘，得到某个连杆的位姿
def link_pose(link, q):                                            # 沿运动链连乘：某个连杆在世界系里的 4×4 变换
    pose = np.eye(4)                                               # 从这个连杆自己出发
    while link in chain:                                           # 一路往父连杆走，直到世界系
        j = chain[link]                                            # 把它接到父连杆上的关节
        step = np.eye(4)                                           # 这一级的变换
        step[:3, :3], step[:3, 3] = rot_rpy(*j["rpy"]), j["xyz"]   # 固定的朝向与位置
        if j["kind"] == "revolute":                                # 转动关节还要再转 q（弧度）
            turn = np.eye(4)                                       # 关节转动的变换
            turn[:3, :3] = rot_axis(j["axis"], q[JOINTS.index(j["name"])])  # 绕转轴转关节角
            step = step @ turn                                     # 先固定变换，再转关节
        pose = step @ pose                                         # 连乘到已经算好的部分上
        link = j["parent"]                                         # 换到父连杆
    return pose  # 连乘的结果

In [ ]:
# 准备（六）：正运动学（二）：夹爪中心的位置；逆运动学的初值
def tcp(q):                                                        # 夹爪中心 = 两个指尖的中点（世界系，米）
    tips = [link_pose(name, q)[:3, 3] for name in ("finger1_tip", "finger2_tip")]  # 两个指尖的位置
    return (tips[0] + tips[1]) / 2                                 # 取中点

Q_INIT = np.radians([0, -20, 45, 45, 0, -8.1])  # 逆运动学的初值：手臂向前、夹爪朝下（夹爪保持起点的 −8.1°）

In [ ]:
# 准备（七）：逆运动学：从目标位置解出五个臂关节角
def solve_ik(target, q_init=Q_INIT, damping=0.01):                 # 数值逆运动学：阻尼最小二乘，雅可比用有限差分
    q = np.clip(q_init, LIMITS[:, 0], LIMITS[:, 1])                # 从初值出发，先夹进限位
    for _ in range(200):                                           # 最多迭代 200 次
        error = target - tcp(q)                                    # 夹爪中心还差多少
        if np.linalg.norm(error) < 1e-3:                           # 差不到 1 毫米就停
            break  # 够近了，结束迭代
        jac = np.zeros((3, 5))                                     # 雅可比：五个臂关节各转一点点，夹爪中心怎么动
        for i in range(5):                                         # 逐个臂关节
            dq = np.zeros(6)                                       # 只动这一个关节
            dq[i] = 1e-5                                           # 转 1e-5 弧度
            jac[:, i] = (tcp(q + dq) - tcp(q)) / 1e-5              # 有限差分
        q[:5] += jac.T @ np.linalg.solve(jac @ jac.T + damping**2 * np.eye(3), error)  # 阻尼最小二乘的一步
        q = np.clip(q, LIMITS[:, 0], LIMITS[:, 1])                 # 每步都夹回限位，免得解出越限的角
    return q.astype(np.float32)                                    # 目标关节角（弧度）

## 一、两种动作模式走到同一个目标

目标是方块中心上方 5 厘米。绝对位置模式一次把目标角发出去；目标增量模式每步按每个关节的上限，分步走到。

In [ ]:
# 建环境：绝对位置模式，固定种子 1，每次都从同一个起点出发
env = gym.make(ENV, obs_mode="rgb", control_mode="pd_joint_pos")  # 动作 = 目标关节角（弧度）
obs, _ = env.reset(seed=1)  # 复位
sim = env.unwrapped  # 仿真内部对象，读真值用
q_start = obs["agent"]["noisy_qpos"][0].numpy()  # 当前六个关节角（弧度）
top = obs["sensor_data"]["top"]["rgb"][0].cpu().numpy()  # 顶部相机的画面，从显卡搬回内存
plt.imsave(OUT / "05-top.png", top)  # 存下来，课件里用

In [ ]:
# 目标：夹爪中心到方块中心上方 5 厘米，关节角用上一讲的逆运动学解出来
cube = sim.item.pose.p[0].numpy()  # 仿真给的方块中心位置（真值，米）
q_goal = solve_ik(cube + [0, 0, 0.05])  # 数值逆运动学：得到目标关节角（弧度）
print("目标关节角（度）：", np.degrees(q_goal).round(1))  # 五个臂关节，加夹爪

In [ ]:
# 绝对位置模式：动作就是目标关节角，每一步都发同一个目标
log_abs = [np.degrees(q_start)]  # 记下每一步的关节角（度），先放起点
for t in range(60):  # 走 60 步，每步 1/30 秒
    obs, *_ = env.step(q_goal[None, :])  # 动作 = 目标关节角（弧度）
    q_now = obs["agent"]["noisy_qpos"][0].numpy()  # 这一步之后的关节角（弧度）
    log_abs.append(np.degrees(q_now))  # 记下来

In [ ]:
# 目标增量模式：动作是归一化的增量（−1 到 1），仿真里再乘以每个关节的每步上限
env.close()  # 关掉绝对位置模式的环境
env = gym.make(ENV, obs_mode="rgb", control_mode="pd_joint_target_delta_pos")  # 换动作模式
obs, _ = env.reset(seed=1)  # 同一个种子，起点相同
sim = env.unwrapped  # 新环境的内部对象
cap = np.array(per_step_limits(JOINTS), dtype=np.float32)  # 每个关节每步最多走多少（弧度）
q_target = q_start.copy()  # 控制器里当前的目标角，从当前关节角开始
log_inc = [np.degrees(q_start)]  # 记下每一步的关节角（度）
for t in range(120):  # 最多走 120 步
    a = np.clip((q_goal - q_target) / cap, -1, 1)  # 还差多少 ÷ 每步上限，夹到 −1…1
    q_target = q_target + a * cap  # 目标角加上这一步的增量
    obs, *_ = env.step(a[None, :])  # 动作是归一化增量
    q_now = obs["agent"]["noisy_qpos"][0].numpy()  # 这一步之后的关节角（弧度）
    log_inc.append(np.degrees(q_now))  # 记下来

In [ ]:
# 数一数各用了几步：五个臂关节都进到目标 1° 以内才算走到
def steps_to_reach(log):  # 第几步第一次走到
    err = np.abs(np.array(log)[:, :5] - np.degrees(q_goal)[:5])  # 每步每个关节的误差（度）
    close = err.max(axis=1) < 1.0  # 这一步最大误差是否小于 1°
    return int(np.argmax(close)) if close.any() else None  # 第一个走到的步数；一直没走到就是 None


n_abs, n_inc = steps_to_reach(log_abs), steps_to_reach(log_inc)  # 两种模式各用了几步
print(f"绝对位置 {n_abs} 步，目标增量 {n_inc} 步")  # 一步 = 1/30 秒
t_abs = np.arange(len(log_abs)) / SIM_FPS  # 横轴：时间（秒）
t_inc = np.arange(len(log_inc)) / SIM_FPS  # 横轴：时间（秒）
lift_abs = np.array(log_abs)[:, 1]  # 绝对位置：肩俯仰的角度
lift_inc = np.array(log_inc)[:, 1]  # 目标增量：肩俯仰的角度
plt.plot(t_abs, lift_abs, color="tab:orange", label="pd_joint_pos")  # 橙色：绝对位置
plt.plot(t_inc, lift_inc, color="tab:blue", label="target_delta")  # 蓝色：目标增量
plt.xlabel("time (s)")  # 横轴名
plt.ylabel("shoulder_lift (deg)")  # 纵轴名
plt.legend()  # 图例
plt.savefig(OUT / "05-modes.png")  # 存下来
plt.show()  # 显示
record = dict(fps=SIM_FPS, joint="shoulder_lift", tolerance_deg=1.0)  # 实验记录：固定信息
record["goal_deg"] = np.degrees(q_goal).tolist()  # 目标关节角（度）
record["absolute"] = dict(q_deg=np.array(log_abs).tolist(), steps=n_abs)  # 绝对位置
record["increment"] = dict(q_deg=np.array(log_inc).tolist(), steps=n_inc)  # 目标增量
json.dump(record, open(OUT / "05-modes.json", "w"))  # 存进 out/05-modes.json，课件里读

## 二、每步上限与最大速度

每步上限由放慢系数、真机实测速度和控制周期算出；量一量增量模式跑出的最大角速度，和算出来的上限对一对。

In [ ]:
# 每步上限 = 放慢系数 × 真机实测速度 × 控制周期，三个量都读自仿真的速度包线
print("放慢系数", SPEED_SCALE, "控制频率", SIM_FPS, "Hz")  # 0.6 与 30 Hz
for name in JOINTS:  # 逐个关节
    omega = REAL_P95_DEG_PER_S[name]  # 真机实测的 95 分位角速度（度每秒）
    cap_deg = SPEED_SCALE * omega / SIM_FPS  # 乘放慢系数，再除以每秒的步数
    print(f"{name:14s} {omega:5.1f} °/s → 每步 {cap_deg:.2f}°")  # 实测速度与每步上限

In [ ]:
# 量一量：每步关节角的变化 × 每秒的步数 = 实际的角速度，对照放慢系数 × 实测速度
speed_inc = np.abs(np.diff(log_inc, axis=0)) * SIM_FPS  # 目标增量模式的角速度（度每秒）
speed_abs = np.abs(np.diff(log_abs, axis=0)) * SIM_FPS  # 绝对位置模式的角速度，对照用
print("关节           增量模式最大   上限   绝对位置最大")  # 表头，单位都是度每秒
for i, name in enumerate(JOINTS[:5]):  # 五个臂关节
    limit = SPEED_SCALE * REAL_P95_DEG_PER_S[name]  # 增量模式的最大速度
    slow, fast = speed_inc[:, i].max(), speed_abs[:, i].max()  # 两种模式各自跑出的最大角速度
    print(f"{name:14s} {slow:8.1f} {limit:8.1f} {fast:10.1f}")  # 一行三个数

In [ ]:
# 对照：关节增量模式把增量加在当前关节角上，同一个目标会怎么走
env.close()  # 关掉上一个环境
env = gym.make(ENV, obs_mode="rgb", control_mode="pd_joint_delta_pos")  # 换成关节增量模式
obs, _ = env.reset(seed=1)  # 同一个种子，起点相同
sim = env.unwrapped  # 新环境的内部对象
q_now = q_start.copy()  # 当前关节角
log_cur = [np.degrees(q_start)]  # 记下每一步的关节角（度）
for t in range(300):  # 最多走 300 步
    a = np.clip((q_goal - q_now) / cap, -1, 1)  # 还差多少 ÷ 每步上限，夹到 −1…1
    obs, *_ = env.step(a[None, :])  # 动作是归一化增量，加在当前关节角上
    q_now = obs["agent"]["noisy_qpos"][0].numpy()  # 这一步之后的关节角（弧度）
    log_cur.append(np.degrees(q_now))  # 记下来
n_cur = steps_to_reach(log_cur)  # 数一数用了几步
print(f"关节增量 {n_cur} 步，目标增量 {n_inc} 步")  # 同一个目标，两种增量模式

## 三、归一化与反归一化

把肩回转的目标角归一化再还原；部署时漏了反归一化，肩回转只转到 0.18° 附近。

In [ ]:
# 归一化：把肩回转的目标 20° 从 −110°…110° 缩到 −1…1
lo, hi = np.degrees(sim.agent.robot.get_qlimits()[0, 0].numpy())  # 肩回转的限位（度）
a = 20.0  # 想要的肩回转角度（度）
a_hat = 2 * (a - lo) / (hi - lo) - 1  # 归一化，就是 2.2 页的式子
print(f"限位 {lo:.0f}° 到 {hi:.0f}°，归一化值 {a_hat:.4f}")  # 归一化值约 0.1818
back = lo + (a_hat + 1) / 2 * (hi - lo)  # 反归一化：把式子倒过来
print(f"反归一化回来 {back:.1f}°")  # 应当回到 20°

In [ ]:
# 部署：模型输出 a_hat。先反归一化再换成弧度发给仿真；漏了反归一化，就是把 0.1818 当成度发出去
env.close()  # 关掉上一个环境
env = gym.make(ENV, obs_mode="rgb", control_mode="pd_joint_pos")  # 回到绝对位置模式
cases = {"反归一化": back, "漏了": a_hat}  # 两种做法：还原出的度数，和没还原的归一化值
frames = {}  # 存两次的顶部相机画面
for label, pan_deg in cases.items():  # 逐种做法
    obs, _ = env.reset(seed=1)  # 复位到同一个起点
    target = obs["agent"]["noisy_qpos"][0].numpy().copy()  # 其他关节保持当前角
    target[0] = np.radians(pan_deg)  # 只改肩回转的目标：度 → 弧度
    for t in range(60):  # 走 60 步
        obs, *_ = env.step(target[None, :])  # 发给仿真
    pan = np.degrees(obs["agent"]["noisy_qpos"][0, 0].item())  # 肩回转最后转到的角度
    print(f"{label}：目标 {pan_deg:.2f}° → 转到 {pan:.2f}°")  # 对比
    frames[label] = obs["sensor_data"]["top"]["rgb"][0].cpu().numpy()  # 顶部相机的画面

In [ ]:
# 两张画面放在一起看
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 左右两张图
for ax, (label, img) in zip(axes, frames.items()):  # 逐张画
    ax.imshow(img)  # 顶部相机画面
    ax.set_title(label)  # 标题：做法
    ax.axis("off")  # 不画坐标轴
plt.savefig(OUT / "05-pan.png")  # 存下来
plt.show()  # 显示
env.close()  # 用完关掉环境